# Perfilado Empírico de Eventos de Uso (`usage_events_stream`)

Este notebook perfila los **120 archivos JSONL (`events_part_0000.jsonl` a `events_part_0119.jsonl`)** ubicados en `data/raw/usage_events_stream/` para fundamentar con evidencia empírica las decisiones de:
1. **Evolución de esquema (`v1` → `v2`)** y tratamiento de campos opcionales (`carbon_kg`, `genai_tokens`).
2. **Completitud y reglas de imputación** (`value`, `unit`).
3. **Unicidad y deduplicación** (`event_id` vs clave natural).
4. **Calidad, cuarentena (`Quarantine`) y detección de anomalías de costo** (`MAD` vs `Z-Score` por `(service, metric)`).
5. **Distribución temporal, desorden entre micro-lotes (*late data*) y dimensionamiento del `Watermark`** en Spark Structured Streaming.
6. **Integridad referencial** con `customers_orgs` y `resources`.

> **Ejecución reproducible:** No requiere credenciales ni rutas de Colab. Funciona tanto desde la raíz del repositorio como desde la carpeta `notebooks/` usando únicamente la biblioteca estándar de Python (y opcionalmente `pandas`).

In [1]:
import csv
import json
import math
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

# Resolución automática de la raíz del repositorio (funciona desde raíz o desde notebooks/)
REPO_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
STREAM_DIR = REPO_ROOT / "data" / "raw" / "usage_events_stream"

files = sorted(STREAM_DIR.glob("events_part_*.jsonl"))
total_bytes = sum(f.stat().st_size for f in files)

records = []
corrupt_lines = 0
lines_per_file = []
all_keys = set()

for batch_idx, fpath in enumerate(files):
    count = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            s = line.strip()
            if not s:
                continue
            count += 1
            try:
                obj = json.loads(s)
                all_keys.update(obj.keys())
                obj["_source_file"] = fpath.name
                obj["_batch_id"] = batch_idx
                obj["_line_no"] = line_no
                obj["_ts"] = datetime.fromisoformat(obj["timestamp"].replace("Z", "+00:00"))
                records.append(obj)
            except Exception:
                corrupt_lines += 1
    lines_per_file.append(count)

N = len(records)
print(f"Directorio del stream: {STREAM_DIR.relative_to(REPO_ROOT).as_posix()}")
print(f"Archivos JSONL encontrados: {len(files)}")
print(f"Tamaño total en disco: {total_bytes:,} bytes ({total_bytes / (1024*1024):.2f} MB)")
print(f"Total de eventos leídos: {N:,}")
print(f"Líneas JSON corruptas: {corrupt_lines}")
print(f"Eventos por archivo (min, max, valores únicos): {min(lines_per_file)}, {max(lines_per_file)}, {set(lines_per_file)}")
print(f"Claves totales observadas ({len(all_keys)}): {sorted(all_keys)}")

Directorio del stream: data/raw/usage_events_stream
Archivos JSONL encontrados: 120
Tamaño total en disco: 12,970,715 bytes (12.37 MB)
Total de eventos leídos: 43,200
Líneas JSON corruptas: 0
Eventos por archivo (min, max, valores únicos): 360, 360, {360}
Claves totales observadas (13): ['carbon_kg', 'cost_usd_increment', 'event_id', 'genai_tokens', 'metric', 'org_id', 'region', 'resource_id', 'schema_version', 'service', 'timestamp', 'unit', 'value']


## 1. Evolución de Esquema (`v1` vs `v2`) y Análisis de Nulos

Verificamos la proporción de versiones de esquema, el momento exacto del cambio de versión y la presencia/nulidad de cada campo:

In [2]:
sv_counts = Counter(r["schema_version"] for r in records)
print("--- DISTRIBUCIÓN DE SCHEMA_VERSION ---")
for sv in sorted(sv_counts):
    ts_sv = [r["_ts"] for r in records if r["schema_version"] == sv]
    print(f"  schema_version={sv}: {sv_counts[sv]:,} eventos ({sv_counts[sv]/N*100:.2f}%) | Rango: {min(ts_sv).isoformat()} -> {max(ts_sv).isoformat()}")

fields = [
    "event_id", "timestamp", "org_id", "resource_id", "service", "region",
    "metric", "value", "unit", "cost_usd_increment", "schema_version", "carbon_kg", "genai_tokens"
]

print("\n--- NULOS Y CLAVES AUSENTES POR VERSIÓN ---")
print(f"{'Campo':<20} | {'Total Nulos/Ausentes':<20} | {'v1 Nulos/Ausentes':<20} | {'v2 Nulos/Ausentes':<20} | {'Clave ausente v1':<16} | {'Clave ausente v2'}")
print("-" * 124)
for k in fields:
    tot_null = sum(1 for r in records if r.get(k) is None)
    v1_null = sum(1 for r in records if r["schema_version"] == 1 and r.get(k) is None)
    v2_null = sum(1 for r in records if r["schema_version"] == 2 and r.get(k) is None)
    v1_abs = sum(1 for r in records if r["schema_version"] == 1 and k not in r)
    v2_abs = sum(1 for r in records if r["schema_version"] == 2 and k not in r)
    print(f"{k:<20} | {tot_null:7,d} ({tot_null/N*100:5.2f}%)     | {v1_null:7,d} ({v1_null/sv_counts[1]*100:5.2f}%)    | {v2_null:7,d} ({v2_null/sv_counts[2]*100:5.2f}%)     | {v1_abs:7,d}          | {v2_abs:7,d}")

v2_genai = [r for r in records if r["schema_version"] == 2 and r["service"] == "genai"]
v2_other = [r for r in records if r["schema_version"] == 2 and r["service"] != "genai"]
print("\nPresencia de genai_tokens en v2 según servicio:")
print(f"  service='genai'      -> {sum(1 for r in v2_genai if r.get('genai_tokens') is not None):,} con genai_tokens / {len(v2_genai):,} totales (100.0% presente)")
print(f"  otros 5 servicios    -> {sum(1 for r in v2_other if r.get('genai_tokens') is not None):5,d} con genai_tokens / {len(v2_other):,} totales (0.0% presente; no aplica)")

print("\nRelación determinística metric -> unit (para imputar los 2,075 nulos de unit):")
for m in sorted({r["metric"] for r in records}):
    u_counts = Counter(r.get("unit") for r in records if r["metric"] == m)
    print(f"  metric={m!r:<18} -> unit={dict(u_counts)}")

--- DISTRIBUCIÓN DE SCHEMA_VERSION ---
  schema_version=1: 10,800 eventos (25.00%) | Rango: 2025-07-03T00:02:00+00:00 -> 2025-07-17T23:56:00+00:00
  schema_version=2: 32,400 eventos (75.00%) | Rango: 2025-07-18T00:01:00+00:00 -> 2025-08-31T23:58:00+00:00

--- NULOS Y CLAVES AUSENTES POR VERSIÓN ---
Campo                | Total Nulos/Ausentes | v1 Nulos/Ausentes    | v2 Nulos/Ausentes    | Clave ausente v1 | Clave ausente v2
----------------------------------------------------------------------------------------------------------------------------
event_id             |       0 ( 0.00%)     |       0 ( 0.00%)     |       0 ( 0.00%)     |       0          |       0
timestamp            |       0 ( 0.00%)     |       0 ( 0.00%)     |       0 ( 0.00%)     |       0          |       0
org_id               |       0 ( 0.00%)     |       0 ( 0.00%)     |       0 ( 0.00%)     |       0          |       0
resource_id          |       0 ( 0.00%)     |       0 ( 0.00%)     |       0 ( 0.00%)     

## 2. Unicidad, Duplicados, Costos Negativos y Spikes (`MAD` vs `Z-Score`)

Analizamos:
- Duplicados por `event_id` y por clave compuesta `(org_id, resource_id, timestamp, metric)`.
- Costos negativos (`cost_usd_increment < 0` y `< -0.01`) que deben enviarse a `Quarantine`.
- Detección de *spikes* de costo agrupando por **`(service, metric)`** (ya que cada métrica tiene una escala tarifaria distinta dentro de un mismo servicio), comparando **Modified Z-Score basado en MAD (`> 3.5`)** frente a **Z-Score clásico (`> 3.0`)**.

In [3]:
def pctile(sorted_vals, p):
    idx = min(len(sorted_vals) - 1, max(0, int(round(p * (len(sorted_vals) - 1)))))
    return sorted_vals[idx]

unique_eids = len({r["event_id"] for r in records})
nat_keys = Counter((r["org_id"], r["resource_id"], r["timestamp"], r["metric"]) for r in records)
nat_dups = {k: v for k, v in nat_keys.items() if v > 1}

print("--- UNICIDAD Y DUPLICADOS ---")
print(f"event_id únicos: {unique_eids:,} de {N:,} ({N - unique_eids} duplicados de event_id)")
print(f"Claves naturales (org_id, resource_id, timestamp, metric) con >1 evento: {len(nat_dups)} claves ({sum(nat_dups.values())} filas)")

neg_all = [r["cost_usd_increment"] for r in records if r["cost_usd_increment"] < 0]
neg_001 = [x for x in neg_all if x < -0.01]
print("\n--- COSTOS NEGATIVOS (QUARANTINE) ---")
print(f"Eventos con cost_usd_increment < 0:     {len(neg_all):3d} ({len(neg_all)/N*100:.2f}%) | min = {min(neg_all):.4f} USD, max = {max(neg_all):.4f} USD")
print(f"Eventos con cost_usd_increment < -0.01: {len(neg_001):3d} ({len(neg_001)/N*100:.2f}%)")
print(f"Eventos entre -0.01 y 0:                {len(neg_all) - len(neg_001):3d} ({(len(neg_all)-len(neg_001))/N*100:.2f}%)")

by_sm = defaultdict(list)
for r in records:
    if r["cost_usd_increment"] >= 0:
        by_sm[(r["service"], r["metric"])].append(r["cost_usd_increment"])

print("\n--- DETECCIÓN DE SPIKES DE COSTO POR (SERVICE, METRIC) (sobre costos >= 0) ---")
print(f"{'Service':<11} | {'Metric':<16} | {'N':>5} | {'Mediana':>7} | {'MAD':>7} | {'p99':>7} | {'Máx':>7} | {'Z > 3.0':>7} | {'ModZ(MAD) > 3.5':>15}")
print("-" * 107)
tot_z, tot_mz = 0, 0
for (svc, met), vals in sorted(by_sm.items()):
    vals.sort()
    med = pctile(vals, 0.5)
    mad = pctile(sorted(abs(x - med) for x in vals), 0.5)
    mean = sum(vals) / len(vals)
    std = math.sqrt(sum((x - mean) ** 2 for x in vals) / len(vals))
    z3 = sum(1 for x in vals if (x - mean) / std > 3.0)
    mz35 = sum(1 for x in vals if mad > 0 and 0.6745 * (x - med) / mad > 3.5)
    tot_z += z3
    tot_mz += mz35
    print(f"{svc:<11} | {met:<16} | {len(vals):5,d} | {med:7.4f} | {mad:7.4f} | {pctile(vals, 0.99):7.4f} | {vals[-1]:8.4f} | {z3:7d} | {mz35:15d}")
print("-" * 107)
print(f"Total spikes detectados -> Z-Score > 3.0: {tot_z} | Modified Z-Score (MAD) > 3.5: {tot_mz}")

--- UNICIDAD Y DUPLICADOS ---
event_id únicos: 43,200 de 43,200 (0 duplicados de event_id)
Claves naturales (org_id, resource_id, timestamp, metric) con >1 evento: 13 claves (27 filas)

--- COSTOS NEGATIVOS (QUARANTINE) ---
Eventos con cost_usd_increment < 0:     216 (0.50%) | min = -154.4608 USD, max = -0.0029 USD
Eventos con cost_usd_increment < -0.01: 211 (0.49%)
Eventos entre -0.01 y 0:                  5 (0.01%)

--- DETECCIÓN DE SPIKES DE COSTO POR (SERVICE, METRIC) (sobre costos >= 0) ---
Service     | Metric           |     N | Mediana |     MAD |     p99 |     Máx | Z > 3.0 | ModZ(MAD) > 3.5
-----------------------------------------------------------------------------------------------------------
analytics   | cpu_hours        | 1,146 |  0.1282 |  0.0708 |  0.4368 |   4.4271 |       5 |               6
analytics   | requests         | 2,063 |  8.3594 |  0.9562 | 12.2055 | 249.3861 |       8 |               9
analytics   | storage_gb_hours | 1,362 |  0.7064 |  0.3825 |  2.0387

## 3. Distribución Temporal, Desorden de Micro-Lotes y Simulación de `Watermark`

En Spark Structured Streaming, al finalizar cada micro-lote $b-1$, el motor actualiza el watermark en función del máximo `event_time` observado hasta ese momento:
$$\text{Watermark}_{b-1} = \max_{i \le b-1}(\text{event\_time}_i) - \text{tolerancia}$$
Por lo tanto, cualquier evento que ingresa en el micro-lote $b$ con un retraso relativo (`lateness`) mayor a la tolerancia ($\max_{i \le b-1}(\text{event\_time}_i) - \text{timestamp} > \text{tolerancia}$) cae detrás del watermark y es descartado por los operadores con estado.

Medimos la dispersión temporal dentro de cada archivo y simulamos exactamente cuántos eventos se descartan para distintos valores de `withWatermark`:

In [4]:
ts_all = [r["_ts"] for r in records]
min_ts, max_ts = min(ts_all), max(ts_all)
ym_counts = Counter(t.strftime("%Y-%m") for t in ts_all)

print("--- COBERTURA TEMPORAL GLOBAL E INTRA-LOTE ---")
print(f"Timestamp mínimo global: {min_ts.isoformat()}")
print(f"Timestamp máximo global: {max_ts.isoformat()}")
print(f"Rango total: {(max_ts - min_ts).total_seconds()/86400:.2f} días ({(max_ts - min_ts).total_seconds()/3600:.2f} horas) | Eventos por día: exactamente {N // 60} (60 x 720 = {N:,})")
print(f"Eventos por mes: {dict(sorted(ym_counts.items()))}")

batch_ts = defaultdict(list)
for r in records:
    batch_ts[r["_batch_id"]].append(r["_ts"])

batch_mins = [min(batch_ts[b]) for b in range(len(files))]
batch_maxs = [max(batch_ts[b]) for b in range(len(files))]
spreads = sorted((batch_maxs[b] - batch_mins[b]).total_seconds() / 86400.0 for b in range(len(files)))

print("\nDispersión interna (max - min) dentro de cada uno de los 120 archivos:")
for b in [0, 1, 119]:
    sp = (batch_maxs[b] - batch_mins[b]).total_seconds() / 86400.0
    print(f"  Batch {b:04d}: min={batch_mins[b].isoformat()} | max={batch_maxs[b].isoformat()} | amplitud={sp:.2f} días")
print(f"  Amplitud intra-archivo (120 archivos): min={spreads[0]:.2f} días, mediana={pctile(spreads, 0.5):.2f} días, max={spreads[-1]:.2f} días")

cum_max = []
cur = None
for b in range(len(files)):
    cur = batch_maxs[b] if cur is None or batch_maxs[b] > cur else cur
    cum_max.append(cur)

lateness_hours = []
for r in records:
    b = r["_batch_id"]
    if b == 0:
        lateness_hours.append(0.0)
    else:
        diff_h = (cum_max[b - 1] - r["_ts"]).total_seconds() / 3600.0
        lateness_hours.append(max(0.0, diff_h))

lat_b1 = sorted(lateness_hours[360:])
print("\n--- PERCENTILES DE DESORDEN TEMPORAL (LATENESS EN LOTES 1..119) ---")
for p in [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]:
    vh = pctile(lat_b1, p)
    print(f"  p{int(p*100):02d}: {vh:7.2f} horas ({vh/24:5.2f} días)")

print("\n--- SIMULACIÓN DE PÉRDIDA DE EVENTOS SEGÚN WATERMARK ---")
print(f"{'Umbral de Watermark':<28} | {'Eventos Descartados':>19} | {'% Descartado':>12} | {'% Retenido':>10}")
print("-" * 78)
for label, h in [
    ("1 hour (diseño Entrega 1)", 1),
    ("6 hours", 6),
    ("12 hours", 12),
    ("24 hours (1 day)", 24),
    ("72 hours (3 days)", 72),
    ("168 hours (7 days)", 168),
    ("336 hours (14 days)", 336),
    ("720 hours (30 days - p50)", 720),
    ("1080 hours (45 days - p75)", 1080),
    ("1344 hours (56 days)", 1344),
    ("1368 hours (57 days - p95)", 1368),
    ("1440 hours (60 days - 100%)", 1440),
]:
    dropped = sum(1 for x in lateness_hours if x > h)
    pct = dropped / N * 100
    print(f"{label:<28} | {dropped:10,d} / {N:,d} | {pct:11.2f}% | {100-pct:9.2f}%")

--- COBERTURA TEMPORAL GLOBAL E INTRA-LOTE ---
Timestamp mínimo global: 2025-07-03T00:02:00+00:00
Timestamp máximo global: 2025-08-31T23:58:00+00:00
Rango total: 60.00 días (1439.93 horas) | Eventos por día: exactamente 720 (60 x 720 = 43,200)
Eventos por mes: {'2025-07': 20880, '2025-08': 22320}

Dispersión interna (max - min) dentro de cada uno de los 120 archivos:
  Batch 0000: min=2025-07-03T00:08:00+00:00 | max=2025-08-31T16:32:00+00:00 | amplitud=59.68 días
  Batch 0001: min=2025-07-03T02:38:00+00:00 | max=2025-08-31T23:51:00+00:00 | amplitud=59.88 días
  Batch 0119: min=2025-07-03T00:26:00+00:00 | max=2025-08-31T17:24:00+00:00 | amplitud=59.71 días
  Amplitud intra-archivo (120 archivos): min=58.97 días, mediana=59.76 días, max=59.98 días

--- PERCENTILES DE DESORDEN TEMPORAL (LATENESS EN LOTES 1..119) ---
  p01:   13.47 horas ( 0.56 días)
  p05:   71.70 horas ( 2.99 días)
  p10:  143.57 horas ( 5.98 días)
  p25:  359.83 horas (14.99 días)
  p50:  720.08 horas (30.00 días)
  p75

## 4. Verificación de Integridad Referencial (`customers_orgs` y `resources`)

Comprobamos que los `org_id` y `resource_id` de los 43.200 eventos crucen contra las tablas maestras, evidenciando por qué **no deben eliminarse los prefijos `res_` y `org_`** en la limpieza de `resources`:

In [5]:
with open(REPO_ROOT / "data" / "raw" / "customers_orgs.csv", "r", encoding="utf-8") as f:
    orgs_master = {row["org_id"] for row in csv.DictReader(f)}

with open(REPO_ROOT / "data" / "raw" / "resources.csv", "r", encoding="utf-8") as f:
    res_master = {row["resource_id"]: row for row in csv.DictReader(f)}

stream_orgs = {r["org_id"] for r in records}
stream_res = {r["resource_id"] for r in records}
res_stripped = {rid.replace("res_", "") for rid in res_master}

mismatches = sum(
    1 for r in records
    if res_master[r["resource_id"]]["org_id"] != r["org_id"]
    or res_master[r["resource_id"]]["service"] != r["service"]
    or res_master[r["resource_id"]]["region"] != r["region"]
)

print("--- INTEGRIDAD REFERENCIAL DEL STREAM ---")
print(f"Organizaciones únicas en usage_events: {len(stream_orgs)} | Coincidencia con customers_orgs.csv (org_*): {len(stream_orgs & orgs_master)} / {len(stream_orgs)} (100.0%)")
print(f"Recursos únicos en usage_events:      {len(stream_res)} | Coincidencia con raw/resources.csv (res_*):  {len(stream_res & set(res_master))} / {len(stream_res)} (100.0%)")
print(f"Coincidencia con resources_limpio.csv sin prefijo 'res_':                                  {len(stream_res & res_stripped)} / {len(stream_res)} (0.0% -> ROMPE JOIN)")
print(f"Inconsistencias de (org_id, service, region) entre eventos y resources.csv:                {mismatches}")

--- INTEGRIDAD REFERENCIAL DEL STREAM ---
Organizaciones únicas en usage_events: 80 | Coincidencia con customers_orgs.csv (org_*): 80 / 80 (100.0%)
Recursos únicos en usage_events:      400 | Coincidencia con raw/resources.csv (res_*):  400 / 400 (100.0%)
Coincidencia con resources_limpio.csv sin prefijo 'res_':                                  0 / 400 (0.0% -> ROMPE JOIN)
Inconsistencias de (org_id, service, region) entre eventos y resources.csv:                0


## 5. Conclusiones Técnicas y Decisiones Derivadas del Perfilado

1. **Corrección del Watermark (`withWatermark`):**
   - Como los 120 archivos de micro-lote contienen eventos barajados a lo largo de los **60 días** (`2025-07-03` al `2025-08-31`) y ya en `events_part_0000.jsonl` el máximo `timestamp` llega al `2025-08-31T16:32:00Z`, fijar `withWatermark('timestamp', '1 hour')` descarta el **99,08 % de los datos (42.801 de 43.200 eventos)**.
   - **Decisión fundamentada:**
     - En las capas **Bronze y Silver** (ingesta, validación, cuarentena y enriquecimiento), el procesamiento es *stateless* por evento o utiliza deduplicación e idempotencia por `event_id` sin descartar eventos históricos tardíos.
     - Para agregaciones *stateful* en **Structured Streaming** sobre estos 120 micro-lotes (modo *backfill/replay* del dataset del TP), el watermark debe configurarse en **`60 days` (`1440 hours`)** para garantizar **0 % de pérdida** (o **`57 days` / `1368 hours`** si se fija un umbral en el percentil 95 de retraso medido).
2. **Evolución de Esquema (`v1` → `v2`):**
   - El corte ocurre exactamente el **`2025-07-18T00:00:00Z`**: 10.800 eventos `v1` (25 %) antes de esa fecha y 32.400 eventos `v2` (75 %) después.
   - En Silver se lee con un único `StructType` explícito que incluye `carbon_kg` (`DoubleType`, nullable) y `genai_tokens` (`LongType`, nullable). Para `v1` quedan en `NULL` (no se imputan con `0` para no distorsionar promedios de emisiones o tokens, salvo en sumas agregadas de Gold donde `SUM` ignora nulos o usa `COALESCE(..., 0)`).
3. **Imputación determinística de `unit` y preservación de `value` nulo:**
   - Los **2.075 nulos en `unit` (4,80 %)** se imputan en Silver de forma 100 % determinística a partir de `metric` (`requests -> 'count'`, `cpu_hours -> 'hours'`, `storage_gb_hours -> 'gb_hours'`).
   - Los **877 nulos en `value` (2,03 %)** tienen `cost_usd_increment` válido; se conservan en Silver con `value = NULL` (sin inventar consumo físico) para no perder el costo financiero.
4. **Regla de Cuarentena (`Quarantine`) y Detección de Spikes:**
   - Los **216 eventos con `cost_usd_increment < 0` (0,50 %)** (211 de ellos `< -0.01`, llegando a `-154,46 USD`) se desvían a `data/quarantine/rejected_events/` con `_rejection_reason = 'negative_cost'` (en telemetría de uso los incrementos de consumo no pueden ser negativos; los créditos comerciales viven en `billing_monthly`).
   - Los **191 spikes de costo** detectados mediante **Modified Z-Score (`MAD > 3.5`) agrupando por `(service, metric)`** se conservan en Silver/Gold marcados con el flag `is_cost_anomaly = True`.
5. **Integridad Referencial (`resources`):**
   - El 100 % de los 400 recursos y 80 organizaciones del stream coinciden exactamente con `raw/resources.csv` y `raw/customers_orgs.csv` usando los prefijos `res_` y `org_`. Quitar esos prefijos en `resources_limpio.csv` reduce el cruce al **0 %**, por lo que en el Paso 2 se restituyen los prefijos `res_` y `org_` en `resources_limpio.csv`.